# Amazon ML Worker - Kaggle Notebook

**Same codebase, different configuration.** This notebook runs a complete worker pipeline on Kaggle.

> **IMPORTANT**: Opening this notebook does NOT automatically consume GPU quota.
> Heavy operations require explicit execution.

## 1. Worker Configuration

Set your worker parameters here. Change these values for each Kaggle session.

In [ ]:
# === WORKER CONFIGURATION ===
WORKER_ID = 0
TOTAL_WORKERS = 1
TASK = 'etl'  # etl | ocr | cv | nlp | embeddings | baseline | inference

# Schema (update on Day-1)
ID_COLUMN = 'id'
TARGET_COLUMN = 'target'
TEXT_COLUMN = 'text'
IMAGE_COLUMN = 'image'
METRIC = 'f1_macro'
SPLIT_METHOD = 'stratified'  # random | stratified | group

# Resource limits
MAX_ROWS = -1  # -1 = unlimited
BATCH_SIZE = 32
SEED = 42

# Module toggles
ENABLE_OCR = False
ENABLE_CV = False
ENABLE_NLP = False
ENABLE_EMBEDDINGS = False
ENABLE_BASELINE = False

## 2. Environment Check

In [ ]:
import sys, os
# Add project root to path
PROJECT_ROOT = '/kaggle/working/amazon-ml-worker'  # adjust if needed
if os.path.exists(PROJECT_ROOT):
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
else:
    sys.path.insert(0, '.')

from src.hardware.detect import detect_hardware, detect_environment_type, print_report
report = detect_hardware()
report['environment_type'] = detect_environment_type()
print_report(report)

## 3. Minimal Dependency Setup

In [ ]:
# Only install what's missing on Kaggle
import subprocess
def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
        print(f'{package}: already installed')
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

install_if_missing('pyyaml', 'yaml')
install_if_missing('tqdm')
install_if_missing('pyarrow')

## 4. Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.config.loader import WorkerConfig
from src.utils.logging import setup_logging, get_logger
from src.utils.seeds import set_seed
from src.utils.paths import ensure_dir
from src.utils.timing import Timer

setup_logging(worker_id=WORKER_ID)
set_seed(SEED)
log = get_logger('kaggle_worker')

config = WorkerConfig(
    worker_id=WORKER_ID, total_workers=TOTAL_WORKERS, task=TASK,
    id_column=ID_COLUMN, target_column=TARGET_COLUMN,
    text_column=TEXT_COLUMN, image_column=IMAGE_COLUMN,
    metric=METRIC, split_method=SPLIT_METHOD,
    max_rows=MAX_ROWS, batch_size=BATCH_SIZE, seed=SEED,
    enable_ocr=ENABLE_OCR, enable_cv=ENABLE_CV,
    enable_nlp=ENABLE_NLP, enable_embeddings=ENABLE_EMBEDDINGS,
    enable_baseline=ENABLE_BASELINE,
)
print(f'Worker {config.worker_id}/{config.total_workers} | Task: {config.task}')

## 5. Path Setup

In [ ]:
# Kaggle paths
INPUT_DIR = '/kaggle/input'  # Competition data
WORKING_DIR = '/kaggle/working'
OUTPUT_DIR = ensure_dir(WORKING_DIR, 'outputs', f'worker_{WORKER_ID}')
CACHE_DIR = ensure_dir(WORKING_DIR, 'cache')

# List input files
if os.path.exists(INPUT_DIR):
    for p in sorted(Path(INPUT_DIR).rglob('*')):
        if p.is_file():
            size_mb = p.stat().st_size / (1024*1024)
            print(f'  {p.relative_to(INPUT_DIR)} ({size_mb:.1f} MB)')

## 6. Data Discovery & Inspection

In [ ]:
# Load data (update path after competition data is attached)
# train_df = pd.read_csv(f'{INPUT_DIR}/competition-name/train.csv')

# For now, use synthetic data if available
if os.path.exists('synthetic_data/train.csv'):
    train_df = pd.read_csv('synthetic_data/train.csv')
    print(f'Loaded synthetic data: {train_df.shape}')
else:
    print('No data found. Attach competition data or generate synthetic data.')
    train_df = None

In [ ]:
if train_df is not None:
    from src.data.inspect import inspect_dataframe, save_inspection_report
    report = inspect_dataframe(train_df, id_column=ID_COLUMN, target_column=TARGET_COLUMN)
    print(f"Shape: {report['shape']}")
    print(f"Columns: {report['columns']}")
    print(f"Missing: {report['missing_values']}")
    print(f"Duplicates: {report['duplicate_rows']}")

## 7-8. Validation Setup & ETL

In [ ]:
if train_df is not None:
    from src.data.split import create_split
    from src.data.validate import check_leakage
    
    train_split, val_split = create_split(
        train_df, method=SPLIT_METHOD,
        target_column=TARGET_COLUMN, seed=SEED
    )
    print(f'Train: {len(train_split)}, Val: {len(val_split)}')
    
    leak = check_leakage(train_split, val_split, id_column=ID_COLUMN)
    print(f'Leakage: {leak["leakage_found"]}')

## 9-13. Worker Processing (OCR / CV / NLP - Optional)

In [ ]:
# Sharding for parallel tasks
if train_df is not None and TOTAL_WORKERS > 1:
    from src.data.shard import get_shard
    shard = get_shard(train_df, WORKER_ID, TOTAL_WORKERS, ID_COLUMN)
    print(f'Shard {WORKER_ID}/{TOTAL_WORKERS}: {len(shard)} rows')
else:
    shard = train_df

In [ ]:
# NLP processing (if enabled)
if ENABLE_NLP and shard is not None and TEXT_COLUMN in shard.columns:
    from src.nlp.text_utils import batch_clean_text, add_text_features
    shard = batch_clean_text(shard, TEXT_COLUMN)
    shard = add_text_features(shard, TEXT_COLUMN)
    print(f'NLP features added: {shard.shape}')

## 14-15. Feature Generation & Baseline

In [ ]:
if ENABLE_BASELINE and train_df is not None:
    from src.features.builder import build_features
    from src.models.baseline import train_baseline
    from src.evaluation.metrics import compute_metrics, format_metrics_report
    
    feat_train = build_features(train_split, config)
    feat_val = build_features(val_split, config)
    
    model, _ = train_baseline(feat_train, config)
    
    feat_cols = [c for c in feat_val.columns if c not in (ID_COLUMN, TARGET_COLUMN)
                 and feat_val[c].dtype in (np.float64, np.float32, np.int64, np.int32)]
    X_val = np.nan_to_num(feat_val[feat_cols].values.astype(np.float32), nan=0.0)
    val_preds = model.predict(X_val)
    metrics = compute_metrics(val_split[TARGET_COLUMN].values, val_preds, METRIC)
    print(format_metrics_report(metrics, METRIC))

## 16-20. Inference & Output Validation

In [ ]:
# Save worker output
if shard is not None:
    shard.to_parquet(OUTPUT_DIR / 'output.parquet', index=False)
    print(f'Output saved: {OUTPUT_DIR / "output.parquet"}')
    print(f'Rows: {len(shard)}')

## 21. Export Artifacts

Download the output directory to merge with other workers' results.

In [ ]:
# List output files
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.name} ({size_kb:.1f} KB)')